**Purpose:** Build the news corpus: GICS sector dictionaries from SEC 10-K filings (populates `01_data/news/10-K/` — excluded from the repo, run the download cells first) and GDELT BigQuery news extraction (needs your own GCP credentials, see `src/secrets_example.py`).

**Inputs:** `news/gics_description.xlsx`, `news/gics_sp500.xlsx`

**Outputs:** `news/gdelt_eda/gkg_eda_by_sector.csv`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT
from src.seeds import SEED_DATA_NEWS_COLLECTION


# News Extraction

The main goal of this notebook is to conduct the data extraction related to news articles and justify the choices made during the process.

After some research [**00004**] and due to some limitations (paywalls, web scraping limitations, missing historical data, API limitations, etc.), the chosen source for news articles is [ProQuest](https://www.proquest.com).

ProQuest is a comprehensive global research platform providing access to a vast range of scholarly content, including dissertations, theses, journals, newspapers, ebooks, and historical archives. Its advanced search technologies and digital tools enable efficient discovery, management, and use of information, supporting rigorous academic research and facilitating library services.

It allows users to search for news articles from a wide variety of sources, including newspapers, magazines, and journals. The platform provides access to both current and historical news content, making it a valuable resource for researchers, students, and professionals seeking information on various topics. It also allows for filtering by date, subject, document type, language, location, and more.

Although, ProQuest requires the use of specific keywords for search purposes, so the methodology adopted in this project aims to go beyond arbitrary term selection. To construct a sector-specific keyword dictionary aligned with the project's main objectives, a data-driven approach was employed using publicly available company ~~descriptions~~ grouped according to the GICS sectors. This method ensures that the resulting vocabulary reflects the actual language used within each industry, avoiding subjective or manual bias in keyword selection.

While GICS is widely used to classify firms into sectors, there is currently no standardized, publicly available dictionary of sector-specific terms suitable for NLP, text mining, or automated classification. To address this gap, an inspiration from the methodology proposed by [**0a38b**], who developed a financial sentiment dictionary based on the analysis of corporate filings, is drawn. Similarly, we utilize company ~~descriptions~~ segmented by sector to identify frequently occurring terms, thereby constructing a tailored sector-specific dictionary.

## Dictionary Construction

1. select on companies for each sectors

2. focus on 10-K filings ([**0a38b**])

3. extract cleaned text from:

    - Item 1: Business Description

    - Item 1A: Risk Factors ([**0a38b**])

    - Item 7: Management’s Discussion & Analysis ([**0a38b**])

4. candidate word extraction: 

    - Word Frequency Analysis ([**0a38b**])

    - TF-IDF Analysis

    - Part-of-Speech Filtering

1. companies selected per sector from SP500 list [**8267f**] and mapped to CIK codes using [**a04a2**]

In [64]:
import pandas as pd
import json

df = pd.read_excel("news/gics_sp500.xlsx")

df["CIK"] = None
with open("news/company_tickers.json", "r") as f:
    company_tickers = json.load(f)
for info in company_tickers.values():
    df.loc[df["Ticker"] == info["ticker"], "CIK"] = info["cik_str"]
print(f"Number of companies with CIK: {df['CIK'].notna().sum()} / {len(df)}")
df.dropna(subset=["CIK"], inplace=True)
df["CIK"] = df["CIK"].apply(lambda x: str(x).zfill(10))

df.sample(3, random_state=SEED_DATA_NEWS_COLLECTION)

Number of companies with CIK: 306 / 511


,Company Name,Ticker,Exchange,GICS Sector,GICS Industry Group,GICS Industry,GICS Sub-Industry,CIK
320,Myriad Genetics Inc,MYGN,Nasdaq,Health Care,Pharmaceuticals and Biotechnology,Biotechnology,Biotechnology,0000899923
274,Lands' End Inc,LE,Nasdaq,Consumer Discretionary,Retailing,Internet & Direct Marketing Retail,Internet & Direct Marketing Retail,0000799288
194,Freeport-McMoRan Inc,FCX,"New York Stock Exchange, Inc.",Materials,Materials,Metals and Mining,Copper,0000831259


In [65]:
df["GICS Sector"].value_counts()

GICS Sector
Consumer Discretionary    71
Financials                37
Industrials               35
Health Care               31
Information Technology    30
Consumer Staples          28
Communication Services    22
Materials                 18
Energy                    16
Utilities                 12
Real Estate                6
Name: count, dtype: int64

2. extract 10-K filings for the selected companies

In [66]:
import requests

def get_10k_html_links(cik_padded):
    headers = {'User-Agent': 'Your Name your.email@example.com'}
    url = f'https://data.sec.gov/submissions/CIK{cik_padded}.json'
    r = requests.get(url, headers=headers)
    data = r.json()
    ten_ks = []

    recent = data['filings']['recent']
    for i, form in enumerate(recent['form']):
        if form == '10-K':
            accession = recent['accessionNumber'][i]
            doc = recent['primaryDocument'][i]
            acc_no_nodash = accession.replace('-', '')
            html_link = f"https://www.sec.gov/Archives/edgar/data/{int(cik_padded)}/{acc_no_nodash}/{doc}"
            ten_ks.append({
                'date': recent['filingDate'][i],
                'link': html_link,
                'accession': accession
            })
    return ten_ks

3. extract relevant content from the filings (item 1, 1A, 7) [problems: mix of ([ or nothing)Item(. or :) and ITEM, and unicode issues vs ascii (ig)]

In [67]:
from bs4 import BeautifulSoup
from nltk.tokenize import TreebankWordTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
import json

# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

# tokenizer = TreebankWordTokenizer()
# lemmatizer = WordNetLemmatizer()
# stop_words = set(stopwords.words('english'))

def extract_text_from_10k(url):
    # Step 1: Load the website
    headers = {'User-Agent': 'Your Name your.email@example.com'}
    response = requests.get(url, headers=headers)

    # Step 2: Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Step 3: Extract all visible text
    text = soup.get_text(separator='\n', strip=True)

    return text.replace("\xa0", " ")

def extract_relevant_sections(text):
    sections = {
        'Item 1': '',
        'Item 1A': '',
        'Item 7': ''
    }
    
    current_section = None
    for line in text.splitlines():
        line = line.strip()
        if line.lower().replace("[", "").startswith('item 1'):
            current_section = 'Item 1'
        elif line.lower().replace("[", "").startswith('item 1a'):
            current_section = 'Item 1A'
        elif line.lower().replace("[", "").startswith('item 7'):
            current_section = 'Item 7'
        elif line.lower().replace("[", "").startswith('item') and any(line.lower().replace("[", "").startswith(f'item {i}') for i in range(2, 17)):
            current_section = None
        
        if current_section:
            line = tokenizer.tokenize(line)
            processed = [lemmatizer.lemmatize(word.lower()) for word in line if word.lower() not in stop_words and word.isalpha()]
            line = ' '.join(processed)
            if line:
                sections[current_section] += line + " "
    
    return sections

# amount_comp, comp_done = len(df), 0
# for _, row in df.iterrows():
#     cik = row['CIK']
#     ten_ks = get_10k_html_links(cik)
#     data = {}
#     for filing in ten_ks:
#         text = extract_text_from_10k(filing['link'])
#         sections = extract_relevant_sections(text)
#         data[filing["link"]] = sections
#     with open(f"news/10-K/{cik}.json", "w") as f:
#         json.dump(data, f)
#     comp_done += 1
#     print(f"Completed {comp_done} / {amount_comp} companies", end="\r")

4. candidate word extraction

- Aggregate 10-K data by GICS sector
- Remove words that appear in all sector documents (just for saving storage)
- Compute TF-IDF scores (document is sector aggregated text) as save TOP-N per sector

In [68]:
# import os

# failed = 0
# done = 0
# files = os.listdir("news/10-K")
# for file in files:

#     # Get sector from CIK
#     cik = file.split(".")[0]
#     try:
#         sector = df.loc[df["CIK"] == cik, "GICS Sector"].values[0]
#     except IndexError:
#         failed += 1
#         continue

#     # Load 10-K data
#     with open(f"news/10-K/{file}", "r") as f:
#         tenk = json.load(f)
    
#     # Load or create sector data
#     if os.path.exists(f"news/GICS/{sector}.json"):
#         with open(f"news/GICS/{sector}.json", "r") as f:
#             sector_data = json.load(f)
#     else:
#         with open(f"news/GICS/{sector}.json", "w") as f:
#             sector_data = {"_meta": {"sector": sector, "#10ks": 0, "ciks": []}}
#             json.dump(sector_data, f)
    
#     # Add 10-K data to sector data
#     if tenk == {}:
#         continue

#     sector_data["_meta"]["ciks"].append(cik)
#     for link, sections in tenk.items():
#         sector_data["_meta"]["#10ks"] += 1

#         for section, content in sections.items():

#             # Create section if not exists
#             if section not in sector_data:
#                 sector_data[section] = {}
            
#             # Verify if content is not empty
#             if content.strip() == "":
#                 continue

#             # Add content to section
#             content_list = content.split()
#             for word in content_list:
#                 if word not in sector_data[section]:
#                     sector_data[section][word] = 0
#                 sector_data[section][word] += 1

#     with open(f"news/GICS/{sector}.json", "w") as f:
#         json.dump(sector_data, f)

#     done += 1
#     print(f"Processed {file} into sector {sector}. ({done}/{len(files)})", end="\r")

# print(f"Failed to find sector for {failed} files.")

In [69]:
# import os
# sectors = [f for f in os.listdir("news/GICS") if f.endswith(".json")]
# words = {}
# for sector_file in sectors:
#     with open(f"news/GICS/{sector_file}", "r") as f:
#         sector_data = json.load(f)
    
#     words[sector_file] = set()
#     for section, content in sector_data.items():
#         if section == "_meta":
#             continue
#         for word in content.keys():
#             words[sector_file].add(word)
# common_words = set.intersection(*words.values())
print(f"Number of common words across all sectors: {4671}")
# for sector_file in sectors:
#     with open(f"news/GICS/{sector_file}", "r") as f:
#         sector_data = json.load(f)
    
#     for section, content in sector_data.items():
#         if section == "_meta":
#             continue
#         for word in common_words:
#             if word in content:
#                 del content[word]
    
#     with open(f"news/GICS/{sector_file}", "w") as f:
#         json.dump(sector_data, f)

Number of common words across all sectors: 4671


In [70]:
# Merge sections into one "count" section
# import os
# sectors = [f for f in os.listdir("news/GICS") if f.endswith(".json")]
# words = {}
# for sector_file in sectors:
#     with open(f"news/GICS/{sector_file}", "r") as f:
#         sector_data = json.load(f)
    
#     sector_data["count"] = {}
#     for section in list(sector_data.keys()):
#         if section == "_meta" or section == "count":
#             continue

#         for word in sector_data[section].keys():
#             sector_data["count"][word] = sector_data["count"].get(word, 0) + sector_data[section][word]

#         del sector_data[section]
    
#     with open(f"news/GICS/{sector_file}", "w") as f:
#         json.dump(sector_data, f)

In [71]:
# Compute TF-IDF scores
# import json
# import os
# import math
# from collections import defaultdict

# # 1. Load all JSON files
# folder_path = "news/GICS"
# json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

# documents = []
# for file in json_files:
#     with open(os.path.join(folder_path, file), 'r') as f:
#         data = json.load(f)
#         documents.append(data['count'])

# # 2. Build vocabulary
# vocab = set()
# for doc in documents:
#     vocab.update(doc.keys())
# vocab = sorted(vocab)

# # 3. Compute TF for each document
# tf_list = []
# for doc in documents:
#     total_terms = sum(doc.values())
#     tf = {term: doc.get(term, 0) / total_terms for term in vocab}
#     tf_list.append(tf)

# # 4. Compute IDF
# N = len(documents)
# idf = {}
# for term in vocab:
#     doc_count = sum(1 for doc in documents if term in doc)
#     idf[term] = math.log(N / (1 + doc_count)) + 1  # +1 smoothing

# # 5. Compute TF-IDF
# tfidf_list = []
# for tf in tf_list:
#     tfidf = {term: tf[term] * idf[term] for term in vocab}
#     tfidf_list.append(tfidf)

# # Convert to pandas DataFrame for display
# import pandas as pd
# df = pd.DataFrame(tfidf_list)
# df.index = [f.split(".")[0] for f in json_files]
# #df

# # 6. Extract top N terms for each document and save to JSON
# top_n = 25
# top_terms_json = {}

# for index, row in df.iterrows():
#     top_terms = row.sort_values(ascending=False).head(top_n).index.tolist()
#     top_terms_json[index] = top_terms

# # Save to JSON file
# with open("news/TFIDF_10K.json", "w") as f:
#     json.dump(top_terms_json, f, indent=2)

Since the results from this methodology are way to *specific* or *technical*, it might be a good ideia to combine with a more general/common-sense dictionary. For this purpose, [**783c1**] will be used, with the same filtering process done before. This dataset contains sub-industry descriptions for each GICS sub-sector, which can be aggregated to the sector level. 

In [72]:
# import pandas as pd

# gics_df = pd.read_excel("news/gics_description.xlsx", header=4)

# # get descriptions per sector and preprocess
# descriptions = {}
# sector = None
# for _, row in gics_df.iterrows():
#     if pd.notna(row['Unnamed: 1']):
#         sector = row['Unnamed: 1']

#     if sector:
#         if sector not in descriptions:
#             descriptions[sector] = ""
        
#         text = str(row['Unnamed: 7'])
#         text = tokenizer.tokenize(text)
#         processed = [lemmatizer.lemmatize(word.lower()) for word in text if word.lower() not in stop_words and word.isalpha()]
#         text = ' '.join(processed)
#         descriptions[sector] += " " + text

# # count words
# sector_word_counts = {}
# for sector, text in descriptions.items():
#     words = text.split()
#     word_count = {}
#     for word in words:
#         word_count[word] = word_count.get(word, 0) + 1
#     sector_word_counts[sector] = word_count

# # apply tdidf
# import math
# from collections import defaultdict

# N = len(sector_word_counts)

# # Step 1: Compute Document Frequencies (df)
# df = defaultdict(int)
# for doc, word_counts in sector_word_counts.items():
#     for word in word_counts.keys():
#         df[word] += 1

# # Step 2: Compute IDF
# idf = {}
# for word, doc_freq in df.items():
#     idf[word] = math.log((1 + N) / (1 + doc_freq)) + 1  # smoothed IDF

# # Step 3: Compute TF-IDF
# tfidf = {}
# for doc, word_counts in sector_word_counts.items():
#     tfidf[doc] = {}
#     total_terms = sum(word_counts.values())
#     for word, count in word_counts.items():
#         tf = count / total_terms
#         tfidf[doc][word] = tf * idf[word]

# # save top 50 words per sector
# top_n = 50
# top_terms_json = {}
# for sector, scores in tfidf.items():
#     top_terms = sorted(scores, key=scores.get, reverse=True)[:top_n]
#     top_terms_json[sector] = top_terms
# with open("news/TFIDF_GICS.json", "w") as f:
#     json.dump(top_terms_json, f, indent=2)

Final dictionaries:

In [73]:
import json
with open("news/TFIDF_10K.json", "r") as f:
    tfidf_10k = json.load(f)
with open("news/TFIDF_GICS.json", "r") as f:
    tfidf_gics = json.load(f)

tfidf_10k.keys(), tfidf_gics.keys()

(dict_keys(['Communication Services', 'Information Technology', 'Consumer Staples', 'Financials', 'Real Estate', 'Industrials', 'Energy', 'Materials', 'Consumer Discretionary', 'Utilities', 'Health Care']),
 dict_keys(['Energy', 'Materials', 'Industrials', 'Consumer Discretionary', 'Consumer Staples', 'Health Care', 'Financials', 'Information Technology', 'Communication Services', 'Utilities', 'Real Estate']))

This limit of N words per sector is somewhat arbitrary, but it is a trade-off between having a comprehensive dictionary and avoiding noise from less relevant terms. It is also important in order to make sure every sector has the same weight when used for news extraction.

In [74]:
%reset -f

## News Scraping

Even though a dictionary 10-K based was constructed and was going to be the main source, based on an quick analysis, I expect that the GICS description based dictionary will be more useful for news articles, since it is more general and common-sense. The 10-K based dictionary has some clear references to specific companies and products that are not likely to appear in news articles, or even if they do, that would mean an impact to the whole sector (e.g., "adobe" for the Information Technology sector, and "monster", "beverage" for the Consumer Staples sector).

I will be using the GICS description based dictionary for the news scraping, but I will keep the 10-K based dictionary for future reference.

Filters to be used:

- Language: English

- Document Type: News

- Publication Title: Wall Street Journal, Wall Street Journal (Online), New York Times (Online), New York Times **????**

Publication title selection rationale: these are the most reputable and widely read newspapers in the US, with a strong focus on business and financial news. They are likely to provide high-quality, relevant articles that can impact stock prices and market sentiment. Also they are the most mentioned in the academic literature, and the only relevant ones available in ProQuest (from the literature mentions point of view).

Actually, after further research, ...

[**93d76**] shows the importance of using besides the headline of the news, the content, which was not avaible for all news, neither fast enought to extract when available. Following this work methodology, I will be using the headline and a snippet available from google news search.

The dictionary will be used to do the search with the following query:
```
word site:wsj.com
word site:nytimes.com
word site:reuters.com
```
with filter per month to avoid too many results at once and be able to have some sort of control over the process in case of errors or interruptions.

In [75]:
import json

# Dictionary from GICS descriptions: top 50 words per sector (TF-IDF based)
with open("news/TFIDF_GICS.json", "r") as f:
    tfidf_gics = json.load(f)

Ideas:

- multiple chrome drivers to speed up the process using selenium -> lead to captcha

- in-browser scraping using javascript -> 

## WSJ

https://www.wsj.com/news/archive/years?mod=wsjfooter but only headlines due to paywall

# GDELT 

requires dictionary OK , not outlet specific ig , only last 10 years ...

In [76]:
# from google.cloud import bigquery
# import os

# # create a GCP service account with BigQuery access and point src/secrets.py at its JSON key
# from src.secrets import GOOGLE_APPLICATION_CREDENTIALS_PATH  # see src/secrets_example.py
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GOOGLE_APPLICATION_CREDENTIALS_PATH


# client = bigquery.Client()

# query = """
# SELECT 
#   DATE,
#   DocumentIdentifier,
#   Quotations
# FROM 
#   `gdelt-bq.gdeltv2.gkg`
# WHERE 
#   REGEXP_CONTAINS(LOWER(Quotations), r'\\boil\\b')
# ORDER BY DATE DESC
# """

# df = client.query(query).to_dataframe()


gdelt has gdeltv2 and normal one

gdeltv2 came up in 2025 but:

"Rest assured that the GDELT 1.0 data streams will be maintained at a minimum through the end of Spring 2015, so your existing applications will continue to work without modification.  At present the GDELT 2.0 data streams only stretch back to late morning February 19, 2015, so those wishing to perform longitudinal analysis will still need to use GDELT 1.0 for historical analysis and GDELT 2.0 for realtime analysis – in late Spring 2015 we will be releasing the entire historical backfile back to 1979 in the GDELT 2.0 format."
https://blog.gdeltproject.org/gdelt-2-0-our-global-world-in-realtime/

**gdelt-bq.gdeltv2**

| Table                      | Purpose                                                                          | Granularity                         | Update Frequency | Typical Size                             | Example Use Case                                          |
| -------------------------- | -------------------------------------------------------------------------------- | ----------------------------------- | ---------------- | ---------------------------------------- | --------------------------------------------------------- |
| **`events`**               | Core event database (who did what to whom, where, when)                          | One row per coded event             | Every 15 min     | Hundreds of millions of rows             | Conflict analysis, protest tracking, cooperation networks |
| **`mentions`**             | Mentions of events in news articles                                              | One row per mention of an event     | Every 15 min     | Billions of rows                         | Media coverage intensity, source-level analysis           |
| **`gkg`**                  | Global Knowledge Graph (context, themes, tone, locations, people, organizations) | One row per document (news article) | Every 15 min     | Billions of rows                         | Thematic / sentiment / co-occurrence analysis             |
| **`gkg_partitioned`**      | Same as `gkg` but partitioned by date                                            | One row per document                | Every 15 min     | More efficient for time-filtered queries | Fast temporal analyses                                    |
| **`events_partitioned`**   | Same data as `events`, but partitioned by date                                   | One row per coded event             | Every 15 min     | Lighter for date-range queries           | Faster longitudinal queries                               |
| **`mentions_partitioned`** | Same as `mentions`, but partitioned                                              | One row per mention                 | Every 15 min     | Largest of all                           | Tracking how events are reported over time                |
| *(Occasionally others)*    | e.g. experimental tables, snapshot archives                                      | Varies                              | Varies           | —                                        | —                                                         |


**best for etf portfolio optimization**

| Table                                       | Best For                                     | Why / Typical Signal                                                                                                                                                                                                 | Limitations for Finance Use                                                            |
| ------------------------------------------- | -------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------- |
| **`gdelt-bq.gdeltv2.gkg_partitioned`**      | ✅ **Best for ETF optimization**              | Contains per-article metadata: *themes*, *locations*, *persons*, *organizations*, and *tone* (sentiment). You can map these to sectors, geographies, or macro topics (e.g. “Energy”, “Defense”, “Emerging Markets”). | Doesn’t include coded “who-did-what” events, so you must aggregate by theme or entity. |
| ~~**`gdelt-bq.gdeltv2.events_partitioned`**~~   | Good for macro / geopolitical event triggers | Captures protests, conflicts, policy changes — can drive risk-on/off or commodity ETFs. Useful for “event-shock” modeling.                                                                                           | Tone is limited; data is sparser for financial topics.                                 |
| ~~**`gdelt-bq.gdeltv2.mentions_partitioned`**~~ | Good for **volume / attention metrics**      | Lets you measure *how much* media coverage a topic or country is getting, which can proxy investor attention or volatility.                                                                                          | You’ll still need GKG or Events to classify what’s being mentioned.                    |


Yes — the GKG (Global Knowledge Graph) is the only GDELT dataset that contains direct textual excerpts or quotations from articles (in the form of V2Counts, V2GCAM, and especially V2Themes and V2Extras fields that can include quoted text or short snippets).

There is the v1: http://data.gdeltproject.org/gkg/index.html (YYYYMMDD.gkg.csv.zip) but the columns are only 

```DATE	NUMARTS	COUNTS	THEMES	LOCATIONS	PERSONS	ORGANIZATIONS	TONE	CAMEOEVENTIDS	SOURCES	SOURCEURLS```

not containing any type of citations or quotes, so the sentiment would be reduced to the TONE generated via the proprietary (GDELT) algorithm.

There is a trade off between using pre-calculated sentiment (TONE) (and more historical data (at least 2013, is there more???? it should be ~2000 or more)) vs extracting quotes and applying a custom sentiment analysis model (and less historical data (2015)). Pre-calculated sentiment is fast and easy to use, but may not capture the nuances of specific quotes or contexts relevant to financial markets. Extracting quotes allows for more tailored sentiment analysis, but requires additional processing and may introduce noise if quotes are not representative.

I will be creating the pipeline for v2 and try to make it modular enough to be adapted to v1 if needed.

plane changes

aftet further though, i decided that since reddit only has data since 2009, but its knda messy until 2011 i would say, or maybe even later, and due to the fact that we have lots of data (daily at least), t might be a good idea to focus on 2015/06/01 until 2025/06/01 (end of reddit) and this makes exactly 10 years, a considerable periodo for anaylis, having all the data for both reddit and news from gdeltv2.

this allows the usage of gdeltv2 without any issues.

the only se não é que há duas inception dates de etfs, uma a 10/2015 e outra em 2018, mas lideremos com isso de forma alguma colocando esticoes no portfolio ou assim, will see

gdelt-bq.gdeltv2.gkg_partitioned will be used, its the same as gdelt-bq.gdeltv2.gkg but with partitioning by date, making queries faster when filtering by date ranges.

In [77]:
from google.cloud import bigquery
import os
import pandas as pd

# ──────────────────────────────
#  SETUP
# ──────────────────────────────
from src.secrets import GOOGLE_APPLICATION_CREDENTIALS_PATH  # see src/secrets_example.py
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GOOGLE_APPLICATION_CREDENTIALS_PATH
client = bigquery.Client()

In [78]:
# q_gkg_dates = """
# SELECT MIN(DATE) AS min_date, MAX(DATE) AS max_date
# FROM `gdelt-bq.gdeltv2.gkg_partitioned`
# """
# gkg_dates = client.query(q_gkg_dates).to_dataframe()
# gkg_dates

20150217234500, 20251030201500

(20150217234500, 20251030201500)

In [79]:
# table = client.get_table("gdelt-bq.gdeltv2.gkg_partitioned")
# preview = client.list_rows(table, max_results=5).to_dataframe()

# preview = preview.drop(columns=["DocumentIdentifier", "Counts", "Themes", "Locations", "Persons", "Organizations", "SharingImage", "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Extras", "Dates", "AllNames", "Amounts", "TranslationInfo", "GCAM", "V2Counts", "SourceCollectionIdentifier"])

# preview = preview.drop(columns=["V2Persons", "V2Organizations"])

# print(preview.columns)

# preview

print("""Index(['GKGRECORDID', 'DATE', 'SourceCommonName', 'V2Themes', 'V2Locations',
       'V2Tone', 'Quotations'],
      dtype='object')""")

Index(['GKGRECORDID', 'DATE', 'SourceCommonName', 'V2Themes', 'V2Locations',
       'V2Tone', 'Quotations'],
      dtype='object')


In [80]:
q_gkg_quotes = """
SELECT
  GKGRECORDID,
  DATE,
  SourceCommonName,     -- Filtrar nomes das sources para US-only ?
  V2Themes,             -- Fazer pesquisa por tema para além de quotes ?
  V2Locations,          -- Filter locations sem US ? 
  V2Tone,               -- V2Tone vs BERT/custom model ?
  Quotations
FROM
  `gdelt-bq.gdeltv2.gkg_partitioned` AS gkg
WHERE
  _PARTITIONDATE BETWEEN '2024-01-01' AND '2024-03-31'
  AND gkg.Quotations IS NOT NULL
  AND gkg.Quotations LIKE '%war%'
ORDER BY
  DATE DESC
LIMIT 100
"""

# Run query
gkg_quotes = client.query(q_gkg_quotes).to_dataframe()

gkg_quotes

E0000 00:00:1765104519.660750 1563022 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,GKGRECORDID,DATE,SourceCommonName,V2Themes,V2Locations,V2Tone,Quotations
0,20240331234500-481,20240331234500,heraldscotland.com,"CRISISLEX_CRISISLEXREC,4517;TAX_FNCACT_HISTORI...","4#Brighton, Brighton And Hove, United Kingdom#...","0.307219662058372,3.53302611367127,3.225806451...","153|150||I love all of Nick Cave albums , but ..."
1,20240331234500-414,20240331234500,mediaite.com,"RELEASE_HOSTAGE,1265;TAX_FNCACT_VICTIMS,1412;C...",1#Israel#IS#IS##31.5#34.75#IS#275;1#Israel#IS#...,"-5.10204081632653,3.06122448979592,8.163265306...",1869|263||May the risen Christ open a path of ...
2,20240331234500-723,20240331234500,abc.net.au,"EPU_POLICY_POLICY,1782;EPU_POLICY_POLICY,3024;...","3#Washington, Washington, United States#US#USD...","-5.79964850615114,1.75746924428823,7.557117750...",3354|39||impose unwarranted unilateral sanctio...
3,20240331233000-490,20240331233000,rrmodelcraftsman.com,"CRISISLEX_CRISISLEXREC,2764;WB_1174_WAREHOUSIN...","3#Boston, Massachusetts, United States#US#USMA...","-1.1166253101737,1.36476426799007,2.4813895781...",668|46|| lumber with a plywood top and four 2 ...
4,20240331233000-456,20240331233000,spokesman.com,"UNGP_FORESTS_RIVERS_OCEANS,374;TAX_DISEASE_BRO...",1#Ukraine#UP#UP##49#32#UP#312;1#Ukraine#UP#UP#...,"-3.07234886025768,2.67591674925669,5.748265609...",528|39||to the city ( of Rome ) and the world#...
...,...,...,...,...,...,...,...
95,20240331180000-609,20240331180000,the-sun.com,"TAX_FNCACT_PRINCESS,600;TAX_FNCACT_PRINCESS,40...","4#London, London, City Of, United Kingdom#UK#U...","3.03354746609565,5.85296216987866,2.8194147037...",968|35||road ahead is looking very positive#10...
96,20240331180000-527,20240331180000,salon.com,"LEADER,126;LEADER,1538;TAX_FNCACT_PRESIDENT,12...","2#Wisconsin, United States#US#USWI##44.2563#-8...","-3.67647058823529,1.83823529411765,5.514705882...",549|39||Never forget our cowards and weaklings...
97,20240331174500-124,20240331174500,yahoo.com,"TAX_FNCACT_CHIEF,1053;TAX_FNCACT_ANALYST,543;T...",None,"-0.895522388059702,0.298507462686567,1.1940298...",1051|147||a tendency to shift away from brande...
98,20240331174500-235,20240331174500,koreatimes.co.kr,"TAX_RELIGION_CHURCH,1947;TAX_DISEASE_FATIGUE,1...","4#Dubai, Dubayy, United Arab Emirates#AE#AE03#...","-1.1703511053316,2.73081924577373,3.9011703511...",1057|41||the walls of selfishness and indiffer...


# EDA for decision support

- for a sample of words:

    - list of SourceCommonName

    - distribution of news per day: filtered vs unfiltered

    - list of themes

In [81]:
# import sqlite3
# from collections import Counter
# import json
# from datetime import datetime, timedelta

# def log_location_filtering_stats(gkg_quotes, word, start_date, end_date, log_path="news/gdelt_eda/location_filtering.jsonl"):
#     a = gkg_quotes["V2Locations"].map(lambda x: 1 if x is not None and "United States" in x else 0)

#     record = {
#         "word": word,
#         "date_range": f"{start_date} - {end_date}",
#         "total": len(a),
#         "United States": int(sum(a))
#     }

#     with open(log_path, "a") as f:
#         f.write(json.dumps(record) + "\n")

# def log_themes(gkg_quotes, log_path="news/gdelt_eda/themes.jsonl"):
#     a = gkg_quotes["V2Themes"]

#     record = {
#         "themes": a.to_list()
#     }

#     with open(log_path, "a") as f:
#         f.write(json.dumps(record) + "\n")

# def update_counts(gkg_quotes, db_path="news/gdelt_eda/sources.db"):
#     """Incrementally update source counts from new gkg_quotes batch."""
#     conn = sqlite3.connect(db_path)
#     cur = conn.cursor()
#     cur.execute("""
#         CREATE TABLE IF NOT EXISTS counts (
#             source TEXT PRIMARY KEY,
#             count INTEGER
#         )
#     """)

#     # Count new batch in memory first (faster than per-row DB inserts)
#     batch_counts = Counter(gkg_quotes["SourceCommonName"])

#     # Use efficient bulk upserts
#     cur.executemany("""
#         INSERT INTO counts (source, count)
#         VALUES (?, ?)
#         ON CONFLICT(source)
#         DO UPDATE SET count = count + excluded.count
#     """, batch_counts.items())

#     conn.commit()
#     conn.close()

# word = "oil"

# start = datetime(2020, 2, 6) # datetime(2015, 6, 1) # BROKE THE QUOTA
# end = datetime(2025, 6, 1)
# delta = timedelta(days=15)

# current = start

# while current < end:
#     start_date = current.strftime("%Y-%m-%d")
#     next_date = (current + delta)
#     end_date = min(next_date, end).strftime("%Y-%m-%d")

#     print(f"Processing {start_date} → {end_date}")

#     q_gkg_quotes = f"""
#     SELECT
#       SourceCommonName,
#       V2Locations,
#       V2Themes
#     FROM
#       `gdelt-bq.gdeltv2.gkg_partitioned` AS gkg
#     WHERE
#       _PARTITIONDATE BETWEEN '{start_date}' AND '{end_date}'
#       AND gkg.Quotations IS NOT NULL
#       AND gkg.Quotations LIKE '%{word}%'
#     """

#     # Run query
#     gkg_quotes = client.query(q_gkg_quotes).to_dataframe()

#     # Your custom functions
#     update_counts(gkg_quotes)
#     log_location_filtering_stats(gkg_quotes, word, start_date, end_date)
#     log_themes(gkg_quotes)

#     # Move to the next interval
#     current = next_date + timedelta(days=1)


https://blog.gdeltproject.org/mapping-the-media-a-geographic-lookup-of-gdelts-sources/
- http://data.gdeltproject.org/blog/2018-news-outlets-by-country-may2018-update/MASTER-GDELTDOMAINSBYCOUNTRY-MAY2018.TXT

~~https://github.com/chrisjvargo/gdelt?tab=readme-ov-file~~

http://data.gdeltproject.org/gdeltv2/masterfilelist.txt

In [82]:
import pandas as pd
import sqlite3

# URL of the dataset
url = "http://data.gdeltproject.org/blog/2018-news-outlets-by-country-may2018-update/MASTER-GDELTDOMAINSBYCOUNTRY-MAY2018.TXT"

# Load the tab-delimited data into a DataFrame
df = pd.read_csv(url, sep="\t", header=None, dtype=str, encoding="latin-1")

# If the second column represents the country code, filter for 'US'
us_df = df[df[1] == "US"]

conn = sqlite3.connect("news/gdelt_eda/sources.db")
df_counts = pd.read_sql_query("SELECT * FROM counts", conn)
conn.close()
df_counts.columns = ['site', 'count']
df_counts = (
    df_counts.groupby('site', as_index=False)['count'].sum()
)

us_df['count'] = us_df[0].map(df_counts.set_index('site')['count'])
us_df['count'] = us_df['count'].fillna(0).astype(int)

us_df = us_df.sort_values(by='count', ascending=False)
us_df = us_df[us_df['count'] > 0]
us_df.head(5)

/var/folders/sz/96b_h5gn3y33k0c7jsvrstmm0000gn/T/ipykernel_54172/4099873586.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  us_df['count'] = us_df[0].map(df_counts.set_index('site')['count'])
/var/folders/sz/96b_h5gn3y33k0c7jsvrstmm0000gn/T/ipykernel_54172/4099873586.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  us_df['count'] = us_df['count'].fillna(0).astype(int)


,0,1,2,count
1667,4-traders.com,US,United States,42840
103002,marketwatch.com,US,United States,37461
79356,iheart.com,US,United States,37118
187111,yahoo.com,US,United States,30381
133042,prnewswire.com,US,United States,26219


In [83]:
q_gkg_quotes = """
SELECT
  GKGRECORDID,
  DATE,
  SourceCommonName,     -- Filtrar nomes das sources para US-only ?
  --V2Themes,             -- Fazer pesquisa por tema para além de quotes ? Existem demasiados temas (>3000 só num mês de análise)
  --V2Locations,          -- Filter locations sem US ? Vamos filtrar antes pelas sources
  --V2Tone,               -- V2Tone vs BERT/custom model ? Custom Model pq oq interessa é o impacto na indústria
  Quotations
FROM
  `gdelt-bq.gdeltv2.gkg_partitioned` AS gkg
WHERE
  _PARTITIONDATE BETWEEN '2024-01-01' AND '2024-03-31'
  AND gkg.Quotations IS NOT NULL
  AND gkg.Quotations LIKE '%war%'
ORDER BY
  DATE DESC
LIMIT 100
"""

# Extracion Pipeline -> to be fine-tuned (filters)

In [ ]:
import random
from datetime import datetime, timedelta

# Define a start and end date
start = datetime(2015, 7, 1)
end = datetime(2025, 7, 1)

# Generate all 15-min interval timestamps between start and end
timestamps = []
current = start
while current <= end:
    timestamps.append(current.strftime("%Y%m%d%H%M%S"))
    current += timedelta(minutes=15)
timestamps = timestamps[1:]

# Pick 3504 random timestamps
random_timestamps = random.sample(timestamps, len(timestamps))[:1]

In [85]:
import requests, zipfile, io
import pandas as pd
import re
import json
import spacy
import random
nlp = spacy.load("en_core_web_sm")

def clean_quotations(text):
    # Remove numbers before/after '|' and hashtags
    text = re.sub(r'\d+\|\d+\|\|', '', text)
    text = re.sub(r'#\d+', '', text)
    
    # Replace multiple spaces and fix spacing around punctuation
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([.,!?;:])', r'\1', text)
    
    return text.strip()

def lemmatize_quotation(text):
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc]

    return " ".join(lemmas)

urls = [ # http://data.gdeltproject.org/gdeltv2/masterfilelist.txt
    f"http://data.gdeltproject.org/gdeltv2/{ts}.gkg.csv.zip" for ts in random_timestamps
    ]

us_sources = us_df[0].to_list()


with open("news/subind_words_count.json", "r") as f:
    words_per_gics_sector = json.load(f)

sector_patterns = {
    sector: r"(?<!\w)(?:" + "|".join(map(re.escape, words)) + r")(?!\w)" #r"\b(?:" + "|".join(map(re.escape, words)) + r")\b"
    for sector, words in words_per_gics_sector.items()
}


# import time
# start_time = time.time()

processed_urls = 0

#EDA = []
EDA2 = {}
for url in urls:
    processed_urls += 1
    print(f"Processing {url} ({processed_urls}/{len(urls)}: {processed_urls/len(urls)*100:.3f}%) ", end="\r")
    r = requests.get(url)
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open(z.namelist()[0]) as f:
                df = pd.read_csv(f, sep='\t', header=None, dtype=str, encoding='ISO-8859-1')
                df.columns = [
                    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName",
                    "DocumentIdentifier", "Counts", "V2Counts", "Themes", "V2Themes",
                    "Locations", "V2Locations", "Persons", "V2Persons", "Organizations",
                    "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage",
                    "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations",
                    "AllNames", "Amounts", "TranslationInfo", "Extras"
                ]
                df = df[["GKGRECORDID", "DATE", "SourceCommonName", "Quotations"]]
                df = df[df["SourceCommonName"].isin(us_sources)]


                df = df[df["Quotations"].apply(lambda x: isinstance(x, str))]
                df["Quotations"] = df["Quotations"].map(clean_quotations)
                df["Quotations"] = df["Quotations"].map(lemmatize_quotation)

                # Filter for by words
                sector_dfs = {}
                #sector_dfs = {"total": len(df)}
                for sector, pattern in sector_patterns.items():

                    sector_dfs[sector] = df[df["Quotations"].str.contains(pattern, na=False, flags=re.IGNORECASE)]
                    
                    sector_dfs[sector] = sector_dfs[sector].drop_duplicates(subset=["SourceCommonName", "Quotations"])

                    
                    sector_dfs[sector] = sector_dfs[sector]["SourceCommonName"].value_counts().to_dict()
                    for source in sector_dfs[sector]:
                        EDA2[source] = EDA2.get(source, {
                            "total": 0,
                            **{s: 0 for s in words_per_gics_sector.keys()}
                        })
                        EDA2[source]["total"] += sector_dfs[sector][source]
                        EDA2[source][sector] += sector_dfs[sector][source]
                #EDA.append({"url": url, **sector_dfs})
    except Exception as e:
        error_message = f"Failed to process {url}: {e}\n"
        with open("errors.log", "a") as log_file:
            log_file.write(error_message)

    # if random.random() < 0.01: # Save progress occasionally
    #     with open("news/gdelt_eda/gkg_eda_by_source.json", "w") as f:
    #         json.dump(EDA2, f, indent=2)
    #     print(f"Saved progress to news/GDELT EDA/gkg_eda_by_source.json")
    #     with open("errors.log", "a") as log_file:
    #         log_file.write(f"MAIN contains {processed_urls} ({processed_urls/len(urls)*100:.2f}%) processed URLs\n")

    # if random.random() < 0.02:
    #     with open("news/GDELT EDA/gkg_eda_by_source_BUP.json", "w") as f:
    #         json.dump(EDA2, f, indent=2)
    #     print(f"Saved progress to news/GDELT EDA/gkg_eda_by_source_BUP.json")
    #     with open("errors.log", "a") as log_file:
    #         log_file.write(f"BUP contains {processed_urls} ({processed_urls/len(urls)*100:.2f}%) processed URLs\n")


# with open("news/gdelt_eda/gkg_eda_by_source.json", "w") as f:
#     json.dump(EDA2, f, indent=2)
# print(f"Final save to news/GDELT EDA/gkg_eda_by_source.json")

#eda_df = pd.DataFrame(EDA)
# with open("errors.log", "a") as log_file:
#     log_file.write(f"Length of EDA dataframe: {len(eda_df)}\n")
#end_time = time.time()
#elapsed_time = end_time - start_time
# with open("errors.log", "a") as log_file:
#     log_file.write(f"Total processing time: {elapsed_time:.2f} seconds\n")
# eda_df.to_csv("news/gdelt_eda/gkg_eda_by_sector.csv", index=False)

### FINAL: MAIN contains 36556 (10.42%) processed URLs

In [86]:
timestamps[0], timestamps[-1]

('20150701001500', '20250701000000')

In [ ]:
# import json

# with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "r") as f:
#     news_settings = json.load(f)
# news_settings["DATE_filter"] = timestamps
# with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "w") as f:
#     json.dump(news_settings, f, indent=2)

In [ ]:
# import json

# with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "r") as f:
#     news_settings = json.load(f)
# news_settings["Quotations_filter"] = {k: list(v.keys()) for k,v in words_per_gics_sector.items()}
# with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "w") as f:
#     json.dump(news_settings, f, indent=2)

In [102]:
import json
with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "r") as f:
    news_settings = json.load(f)

news_settings.keys()

dict_keys(['SourceCommonName_filter', 'DATE_filter', 'Quotations_filter'])

# ESCREVER METODOLOGIA NA TESE CHAPTER DATA

- uso do gdeltv2 gkg

- devido ao enorme volume foram precisos filtros

- filtro das colunas para gkrecordid, date, sourcecommonname, quotations

- filtro da sourcecommonname para exclusivamente us sources ([link](http://data.gdeltproject.org/blog/2018-news-outlets-by-country-may2018-update/MASTER-GDELTDOMAINSBYCOUNTRY-MAY2018.TXT))

- para ter noticias relevantes aos temas, THEMES tem demasiados tipos, entao tem de ser por palavras presentes nas quotations (limpou-se o numero das quotations e isso para ficar com texto normal)

- baseado nos subindustries descriptions to gics, llms geram para cada up to 10 words. o top 30 palavras por setor passam a ser as keywords usadas para filtrar o gdelt. estas palavras foram lematizadas para ter melhores resultados. regex foi usado para a pesquisa.

- baseado numa amostra de aprox 10% do gdelt, foram guardaddos o numero de noticias por sourcecommonname para analisar que fontes noticiosas para se ter uma ideia de q fontes sao mais relevantes por setor, com o objetivo de filtrar ainda mais o gdelt

- este resultado permite selecionar uma % de noticias por setor. assim foram escolhidas as top sources para ter 10% das noticias por setor, mantendo a proporcao entre setores e uma amostra das noticias relvante, reduzindo o tamanho do dataset (news aux gdelt eda.ipynb)

extras e justificacoes e afins durante o notebook, reler qnd escrever na tese